# **`robopianist` 教程**

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/robopianist/blob/main/tutorial.ipynb)

> 注意：本文件是 tutorial.ipynb 的中文注释版。如果在本地运行，安装单元格可以跳过，直接从 "导入模块" 部分开始执行。

> Note：如果你之前下载过 twinkle_twinkle_actions.npy 到了 examples/ 文件夹下，记得修改 Policy 类的加载路径。

> <p><small><small>Copyright 2023 The RoboPianist Authors.</small></p>
> <p><small><small>Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at <a href="http://www.apache.org/licenses/LICENSE-2.0">http://www.apache.org/licenses/LICENSE-2.0</a>.</small></small></p>
> <p><small><small>Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.</small></small></p>

# 在 Colab 上安装 `robopianist`（本地用户可跳过此节）

In [ ]:
# @title 点击运行以安装 robopianist（仅 Colab 用户需要）
# 如果你在本地运行，且已经 pip install robopianist，请跳过此单元格！
from IPython.display import clear_output
import subprocess

# 检查是否有 GPU 可用
if subprocess.run("nvidia-smi").returncode:
    raise RuntimeError(
        "无法与 GPU 通信。"
        "请确保你使用的是 GPU Colab 运行时。"
        "前往 Runtime 菜单，选择 Change runtime type。"
    )

# 安装系统依赖（声音字体、MuJoCo 等）
!bash <(curl -s https://raw.githubusercontent.com/google-research/robopianist/main/scripts/install_deps.sh) --no-soundfonts --no-menagerie

print("正在安装 robopianist...")
%pip install -q robopianist>=1.0.6

# 设置 MuJoCo 渲染后端为 EGL（Colab 无显示器）
%env MUJOCO_GL=egl

clear_output()
!echo 已安装 $(robopianist --version)

# 导入模块

In [ ]:
# @title 导入本教程需要的所有模块
from IPython.display import HTML
from base64 import b64encode
import numpy as np
from robopianist.suite.tasks import self_actuated_piano  # 自驱动钢琴任务
from robopianist.suite.tasks import piano_with_shadow_hands  # 双手控制钢琴任务
from dm_env_wrappers import CanonicalSpecWrapper
from robopianist.wrappers import PianoSoundVideoWrapper  # 录制视频和音频的包装器
from robopianist import music  # 音乐加载模块
from mujoco_utils import composer_utils  # 环境构建工具
import dm_env  # DeepMind 环境接口

In [ ]:
# @title 辅助函数


# 在 notebook 中播放视频的函数
# 参考: https://stackoverflow.com/a/60986234
def play_video(filename: str):
    """读取 MP4 文件并以 base64 编码后嵌入 HTML 播放。"""
    mp4 = open(filename, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    return HTML(
        """
  <video controls>
        <source src="%s" type="video/mp4">
  </video>
  """
        % data_url
    )

# 自驱动钢琴任务（Self-actuated Piano）

钢琴按键自行按下，无需机器人手控制。适合快速验证 MIDI 文件和钢琴模型。

In [ ]:
# 创建自驱动钢琴任务
task = self_actuated_piano.SelfActuatedPiano(
    midi=music.load("TwinkleTwinkleRousseau"),  # 加载内置的《小星星》MIDI
    change_color_on_activation=True,             # 按键时改变颜色
    trim_silence=True,                           # 裁剪首尾静音
    control_timestep=0.01,                       # 控制步长（秒）
)

# 构建 MuJoCo 环境
env = composer_utils.Environment(
    recompile_physics=False, task=task, strip_singleton_obs_buffer_dim=True
)

# 添加视频录制包装器
env = PianoSoundVideoWrapper(
    env,
    record_every=1,               # 每一帧都记录
    camera_id="piano/back",       # 使用后置摄像头
    record_dir=".",               # 视频保存到当前目录
)

In [ ]:
# 查看动作空间：每个钢琴键 + 延音踏板
action_spec = env.action_spec()
min_ctrl = action_spec.minimum  # 动作最小值
max_ctrl = action_spec.maximum  # 动作最大值
print(f"动作维度: {action_spec.shape}")

In [ ]:
# 查看观测空间
print("观测内容:")
timestep = env.reset()
dim = 0
for k, v in timestep.observation.items():
    print(f"\t{k}: 形状={v.shape} 类型={v.dtype}")
    dim += np.prod(v.shape)
print(f"观测总维度: {dim}")

In [ ]:
# Oracle 策略：直接根据目标状态决定按键
# "目标"观测告诉我们哪些键需要被按下/释放
class Oracle:
    def __call__(self, timestep: dm_env.TimeStep) -> np.ndarray:
        if timestep.reward is not None:
            assert timestep.reward == 0
        # 获取下一个时间步的目标状态
        goal = timestep.observation["goal"][: task.piano.n_keys]
        key_idxs = np.flatnonzero(goal)  # 需要按下的键的索引
        # 需要按下的键：设置为最大动作值
        # 需要释放的键：设置为最小动作值
        action = min_ctrl.copy()
        action[key_idxs] = max_ctrl[key_idxs]
        # 延音踏板动作
        action[-1] = timestep.observation["goal"][-1]
        return action

In [ ]:
# 使用 Oracle 策略运行一个 episode
policy = Oracle()

timestep = env.reset()
while not timestep.last():  # 直到 episode 结束
    action = policy(timestep)
    timestep = env.step(action)

In [ ]:
# 播放录制的视频
play_video(env.latest_filename)

# 双手控制钢琴任务（Piano with Shadow Hands）

使用两只 Shadow Hand 仿生灵巧手来控制钢琴，这是完整的机器人弹琴任务。

In [ ]:
# 创建双手控制钢琴任务
task = piano_with_shadow_hands.PianoWithShadowHands(
    change_color_on_activation=True,      # 按键时改变颜色
    midi=music.load("TwinkleTwinkleRousseau"),  # 加载《小星星》
    trim_silence=True,                     # 裁剪静音段
    control_timestep=0.05,                 # 控制步长
    gravity_compensation=True,             # 重力补偿（手不下坠）
    primitive_fingertip_collisions=False,  # 简化手指碰撞检测
    reduced_action_space=False,            # 是否使用简化动作空间
    n_steps_lookahead=10,                  # 前瞻步数（用于奖励计算）
    disable_fingering_reward=False,        # 禁用指法奖励
    disable_forearm_reward=False,          # 禁用前臂奖励
    disable_colorization=False,            # 禁用键盘变色
    disable_hand_collisions=False,         # 禁用手部碰撞
    attachment_yaw=0.0,                    # 手的安装偏航角
)

# 构建 MuJoCo 环境
env = composer_utils.Environment(
    task=task, strip_singleton_obs_buffer_dim=True, recompile_physics=False
)

# 添加视频录制包装器
env = PianoSoundVideoWrapper(
    env,
    record_every=1,
    camera_id="piano/back",
    record_dir=".",
)

# 规范观测格式
env = CanonicalSpecWrapper(env)

In [ ]:
# 查看动作空间（两只手的所有关节）
action_spec = env.action_spec()
print(f"动作维度: {action_spec.shape}")

In [ ]:
# 查看观测空间
timestep = env.reset()
dim = 0
for k, v in timestep.observation.items():
    print(f"\t{k}: 形状={v.shape} 类型={v.dtype}")
    dim += int(np.prod(v.shape))
print(f"观测总维度: {dim}")

In [ ]:
# 下载预训练策略的动作序列
# 如果文件已存在于 examples/ 目录，可跳过此下载
!wget https://github.com/google-research/robopianist/raw/main/examples/twinkle_twinkle_actions.npy > /dev/null 2>&1


class Policy:
    """预训练回放策略：按顺序播放预先录制的动作序列。"""
    def __init__(self) -> None:
        self.reset()

    def reset(self) -> None:
        """重置动作索引并加载动作文件。"""
        self._idx = 0
        # 如果在本地运行且文件在 examples/ 下，改为:
        # self._actions = np.load("examples/twinkle_twinkle_actions.npy")
        self._actions = np.load("twinkle_twinkle_actions.npy")

    def __call__(self, timestep: dm_env.TimeStep) -> np.ndarray:
        """返回当前时间步的动作，忽略观测。"""
        del timestep  # 不使用
        actions = self._actions[self._idx]
        self._idx += 1
        return actions

In [ ]:
# 使用预训练策略运行一个 episode
policy = Policy()

timestep = env.reset()
while not timestep.last():
    action = policy(timestep)
    timestep = env.step(action)

In [ ]:
# 播放机器人弹奏的视频
play_video(env.latest_filename)

# 总结

本教程涵盖了两个核心任务：

1. **自驱动钢琴任务** — 键盘自动按下，无机器人手，适合验证 MIDI 和钢琴模型
2. **双手控制钢琴任务** — 完整任务，用两只 Shadow Hand 弹琴

关键概念：
- **动作空间**：每个钢琴键对应一个执行器，外加延音踏板
- **观测空间**：包含目标状态（goal）、关节位置、速度、触觉等
- **Oracle 策略**：直接根据目标观测生成动作，完美演奏但不具泛化性
- **预训练策略**：从文件中加载录制的动作序列回放

更多信息请参考: https://github.com/google-research/robopianist